## Part 1. Coco Person & Threat Dataset
This part just downloads the coco datasets subset of persons & threat-related classes and converts this subset into a YOLO usable format then puts it in the final dataset.

In [ ]:
import os
import json
import shutil
from tqdm import tqdm


# base_dir = Full COCO dataset path
base_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco"
# output_dir = Path to the final dataset
output_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-combined-dataset"

# COCO ID to YOLO ID mapping for only the COCO classes we want
coco_id_to_yolo_id = {
    1: 0,     # Person
    8: 1,     # Truck
    39: 2,    # Baseball bat
    49: 4,    # Knife
    27: 7,    # Backpack
    33: 8,    # Suitcase
    87: 9     # Scissors
}

target_classes = set(coco_id_to_yolo_id.keys())
splits = ['train', 'val']

In [ ]:
# Convert COCO annotations to YOLO format
def coco_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    x_c = (x + w / 2) / img_w
    y_c = (y + h / 2) / img_h
    return [x_c, y_c, w / img_w, h / img_h]

In [ ]:
# Performs the conversion for a given split
def process_split(split):
    print(f"Processing {split} split...")
    
    # Input paths
    anno_path = os.path.join(base_dir, "annotations", f"instances_{split}2017.json")
    image_dir = os.path.join(base_dir, "images", f"{split}2017")
    
    # Output paths
    out_img_dir = os.path.join(output_dir, "images", split)
    out_lbl_dir = os.path.join(output_dir, "labels", split)
    
    with open(anno_path) as f:
        coco = json.load(f)
    
    imgs = {img["id"]: img for img in coco["images"]}
    anns = [ann for ann in coco["annotations"] if ann["category_id"] in target_classes]
    
    
    labels_by_image = {}
    for ann in anns:
        img_id = ann["image_id"]
        img = imgs[img_id]
        file_name = img["file_name"]
        img_w, img_h = img["width"], img["height"]
        
        # Uses the pre-defined coco_to_yolo function to convert the bounding box
        yolo_box = coco_to_yolo(ann["bbox"], img_w, img_h)
        yolo_class = coco_id_to_yolo_id[ann["category_id"]]
        label_line = [yolo_class] + yolo_box

        if file_name not in labels_by_image:
            labels_by_image[file_name] = []
        labels_by_image[file_name].append(label_line)

    for file_name, labels in tqdm(labels_by_image.items()):
        src_img = os.path.join(image_dir, file_name)
        dst_img = os.path.join(out_img_dir, file_name)
        dst_txt = os.path.join(out_lbl_dir, file_name.replace(".jpg", ".txt"))

        if not os.path.exists(src_img):
            continue
        
        # Copy the image and create the label file to the output directory
        shutil.copyfile(src_img, dst_img)
        with open(dst_txt, "w") as f:
            for label in labels:
                f.write(" ".join([f"{x:.6f}" for x in label]) + "\n")

In [ ]:
# Runs the conversion for each split
for split in splits:
    process_split(split)

print("Done: COCO subset converted to YOLO format.")

Processing train split...


100%|██████████| 69907/69907 [00:49<00:00, 1398.44it/s]


Processing val split...


100%|██████████| 2920/2920 [00:03<00:00, 918.48it/s] 


✅ Done: COCO subset converted to YOLO format.


# Open Images Threat Dataset
Downloads and converts the threat subset selected of the Open Images V7 dataset into a YOLO friendly format.

In [ ]:
import os
import fiftyone as fo
import fiftyone.zoo as foz

# Classes for the final dataset
final_classes = [
    "Person", "Truck", "Baseball bat", "Rifle", "Knife",
    "Shotgun", "Handgun", "Backpack", "Suitcase", "Scissors"
]

# Classes from OpenImages
openimages_classes = [
    "Truck", "Baseball bat", "Rifle", "Knife",
    "Shotgun", "Handgun", "Backpack", "Suitcase", "Scissors"
]

output_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-combined-dataset"

# Create output directories if they don't exist
for split in ["train", "validation", "test"]:
    os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

class_name_to_yolo_id = {name: final_classes.index(name) for name in openimages_classes}

# Download the OpenImages dataset
dataset = foz.load_zoo_dataset(
    "open-images-v7",
    splits=["train", "validation", "test"],
    classes=openimages_classes,
    label_types=["detections"],
    shuffle=True,
    seed=51,
    dataset_name="open-images-threat-detection-v2"  # Called v2 as it is the second version with less threat classes in it
)

print("Downloaded OpenImages threat subset")

# Convert OpenImages to YOLO format
for split in ["train", "validation", "test"]:
    view = dataset.match_tags(split)
    for sample in view:
        img = sample.filepath
        detections = sample.ground_truth.detections
        metadata = sample.metadata

        if metadata is None:
            print(f"Skipping {img}: missing metadata")
            continue

        img_width = metadata.width
        img_height = metadata.height
        label_lines = []

        for det in detections:
            class_name = det.label
            if class_name not in class_name_to_yolo_id:
                continue

            class_id = class_name_to_yolo_id[class_name]
            x, y, w, h = det.bounding_box
            x_center = x + w / 2
            y_center = y + h / 2
            label_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

        if label_lines:
            img_filename = os.path.basename(img)
            txt_filename = os.path.splitext(img_filename)[0] + ".txt"

            img_out_path = os.path.join(output_dir, "images", split, img_filename)
            label_out_path = os.path.join(output_dir, "labels", split, txt_filename)

            print(f"Copying image to: {img_out_path}")
            if not os.path.exists(img_out_path):
                os.system(f"cp '{img}' '{img_out_path}'")

            print(f"Writing label to: {label_out_path}")
            with open(label_out_path, "w") as f:
                f.write("\n".join(label_lines))

print("OpenImages conversion to YOLO format completed.")

Necessary images already downloaded
Existing download of split 'train' is sufficient
Necessary images already downloaded
Existing download of split 'validation' is sufficient
Necessary images already downloaded
Existing download of split 'test' is sufficient
Loading existing dataset 'open-images-threat-detection'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use
✅ Downloaded OpenImages threat subset
Copying image to: /Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-combined-dataset/images/train/5a3a0410b83c4bdc.jpg
Writing label to: /Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-combined-dataset/labels/train/5a3a0410b83c4bdc.txt
Copying image to: /Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-combined-dataset/images/train/110bde906a54d765.jpg
Writing label to: /Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/

## Final Dataset Verification
This checks through the final dataset to see the class splits of labels per split.

In [1]:
import os
import collections


base_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-combined-dataset"
splits = ["train", "val", "test"]

# Classes for the final dataset
final_classes = [
    "Person", "Truck", "Baseball bat", "Rifle", "Knife",
    "Shotgun", "Handgun", "Backpack", "Suitcase", "Scissors"
]

valid_class_ids = set(range(len(final_classes)))

# Store label box counts and image counts
counts_per_split = {split: collections.Counter() for split in splits}
image_counts = {}

# Track bad labels
invalid_labels = []

# Check the labels and images for each split in each class of the final dataset
for split in splits:
    label_dir = os.path.join(base_dir, "labels", split)
    image_dir = os.path.join(base_dir, "images", split)

    # Count images
    image_count = len([f for f in os.listdir(image_dir) if f.endswith(".jpg")])
    image_counts[split] = image_count

    # Parse label files
    for file in os.listdir(label_dir):
        if not file.endswith(".txt"):
            continue
        path = os.path.join(label_dir, file)
        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    try:
                        class_id = int(float(parts[0]))
                        if class_id in valid_class_ids:
                            counts_per_split[split][class_id] += 1
                        else:
                            invalid_labels.append((split, file, class_id))
                    except:
                        invalid_labels.append((split, file, "non-numeric"))
                else:
                    invalid_labels.append((split, file, "malformed"))

# Display class-based counts for each split
print("\nClass ID distribution across all labels:\n")
for split in splits:
    print(f"{split.upper()} (Images: {image_counts[split]})")
    for class_id in sorted(counts_per_split[split]):
        class_name = final_classes[class_id]
        count = counts_per_split[split][class_id]
        print(f"  Class ID {class_id:2} ({class_name:14}): {count} labels")
    print()

# Show invalid entries if any exist
if invalid_labels:
    print("Found INVALID label entries:")
    for entry in invalid_labels:
        print(f"  Split: {entry[0]}, File: {entry[1]}, Issue: {entry[2]}")
else:
    print("All labels are clean! Only allowed IDs (0–9) are present.")


Class ID distribution across all labels:

TRAIN (Images: 83498)
  Class ID  0 (Person        ): 262465 labels
  Class ID  1 (Truck         ): 22108 labels
  Class ID  2 (Baseball bat  ): 4504 labels
  Class ID  3 (Rifle         ): 2540 labels
  Class ID  4 (Knife         ): 8620 labels
  Class ID  5 (Shotgun       ): 580 labels
  Class ID  6 (Handgun       ): 727 labels
  Class ID  7 (Backpack      ): 9936 labels
  Class ID  8 (Suitcase      ): 6822 labels
  Class ID  9 (Scissors      ): 1880 labels

VAL (Images: 3454)
  Class ID  0 (Person        ): 11004 labels
  Class ID  1 (Truck         ): 769 labels
  Class ID  2 (Baseball bat  ): 168 labels
  Class ID  3 (Rifle         ): 135 labels
  Class ID  4 (Knife         ): 406 labels
  Class ID  5 (Shotgun       ): 54 labels
  Class ID  6 (Handgun       ): 24 labels
  Class ID  7 (Backpack      ): 403 labels
  Class ID  8 (Suitcase      ): 336 labels
  Class ID  9 (Scissors      ): 37 labels

TEST (Images: 1599)
  Class ID  1 (Truck    